In [ ]:
import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from nltk import word_tokenize, sent_tokenize
import sys
import os
from pathlib import Path
import tiktoken
project_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(project_root))
from Components.MultiBlockDecoder import CompleteDecoderBlock

In [22]:
with open(r'../tiny shakespeare.txt', 'r') as file :
    text = file.read()
text
enc = tiktoken.get_encoding("r50k_base")
enc.n_vocab
encoded_corpus = enc.encode(text)

In [ ]:
class DatasetShakespiere(Dataset):
    def __init__(self, chunk_length, token_corpus):
        self.token_corpus = token_corpus
        self.chunk_length = chunk_length
    def __len__(self):
        return (len(self.token_corpus)-1)//self.chunk_length
    
    def __getitem__(self, index):
        start = index * self.chunk_length
        end = start + self.chunk_length + 1
        
        window = self.token_corpus[start : end ]
        input = torch.tensor(window[:-1]) 
        target = torch.tensor(window[1:])
        
        return input, target

dataset = DatasetShakespiere(128, encoded_corpus)
data_loader = DataLoader( dataset, pin_memory=True)

In [25]:
n_blocks = 2 
num_heads = 2
vocab_count = enc.n_vocab
embed_dim = 512
ffo_neurons = 1024
model = CompleteDecoderBlock(n_blocks, num_heads, vocab_count, embed_dim, ffo_neurons, False)


In [27]:
a,b = next(iter(data_loader))

print(enc.decode(a))
           

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be


In [28]:
y = model(a)

AttributeError: 'list' object has no attribute 'shape'